In [ ]:
# ruff: noqa: F401
import functools
import subprocess
from pathlib import Path

import board
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.mcp import MCPTools
from dotenv import load_dotenv
from mcp import StdioServerParameters
from rich.console import Console
from rich.markdown import Markdown

load_dotenv(override=True)
console = Console()

### Create the Agent


In [4]:
MODEL = "gpt-5.4-nano"
model = OpenAIChat(MODEL)

agent = Agent(model=model, instructions="You are a friendly assiatant. Answer in short sentences")

### Run the model


In [7]:
result = await agent.arun("Say Hello in Bengali")
console.print(Markdown(result.content))

Hello in Bengali is “হ্যালো”.

### Using SQLite


In [9]:
board.reset_board()
board.add_goal(
    """
    Read notes.txt, read its contents and convert into Bengali, and write to bengali.txt.
    Overwrite the file if it already exists.
    """
)
board.show_board()
board.list_todos()

Goal #1: 
    Read notes.txt, read its contents and convert into Bengali, and write to bengali.txt.
    Overwrite the file if it already exists.

[{'id': 1,
  'parent_id': None,
  'task': '\n    Read notes.txt, read its contents and convert into Bengali, and write to bengali.txt.\n    Overwrite the file if it already exists.\n    ',
  'status': 'pending',
  'result': ''}]

### Using tools


In [10]:
def show_todos() -> list[dict]:
    """List every tool in the todo list. A goal has a parent id is None. A step has the parent_id set to it's goal_id"""
    return board.list_todos()


def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps

    Args:
        goal_id: The id of the goal to decompose
        steps: Short description of the steps to be taken in order

    """
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}


def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (step or goal) as done and add a short summary

    Args:
        task_id: The id of the todo tak to be done
        result: A short summary of the task completed
    """
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "Done"}

In [11]:
board_agent = Agent(
    model=model, instructions="You help manage a shared todo board", tools=[show_todos, plan_steps, complete_task]
)

result = await board_agent.arun("What are the tasks on the board right now, and what is the status of each task")

console.print(Markdown(result.content))

Here’s what’s on the board right now:                                                                              

 • Task #1 (goal): “Read notes.txt, read its contents and convert into Bengali, and write to bengali.txt. Overwrite
   the file if it already exists.”                                                                                 
    • Status: Pending

### Using MCP


In [ ]:
workspace = Path("workspace").resolve()

server = StdioServerParameters(
    command="npx", args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)], cwd=str(workspace)
)

async with MCPTools(server_params=server, timeout_seconds=60) as filesystem:
    file_agent = Agent(
        model=model,
        instructions="""You can read and write files in your workspace.
        Use the tools for accomplish the tasks assigned to you""",
        tools=[filesystem],
    )

    result = await file_agent.arun("Read notes.txt and summarize in one sentence")


console.print(Markdown(result.content))

The notes describe a collaborative effort to build a small language tutor by having each team member complete one  
task from a shared board carefully and mark it as done.

### Agent in loop


In [16]:
INSTRUCTIONS = (
    "You maintain a todo board."
    "When asked to plan, call show_todos to inspect the board. "
    "For each pending goal without steps, break it into an ordered "
    "checklist and call plan_steps to save it. "
    "Do not duplicate existing steps. "
    "Only mark tasks complete after the work has actually been performed."
)

board.reset_board()
goal_id = board.add_goal(
    "Read notes.txt, translate its contents into Bengali and write it to bengali.txt."
    "Please overwrite if the file already exists"
)
board.claim_todo(goal_id)

async with MCPTools(server_params=server, timeout_seconds=60) as filesystem:
    worker = Agent(model=model, instructions=INSTRUCTIONS, tools=[show_todos, plan_steps, complete_task, filesystem])
    await worker.arun("Please work with the pending goal on the board")


console.print(Markdown(result.content))

The notes describe a collaborative effort to build a small language tutor by having each team member complete one  
task from a shared board carefully and mark it as done.